In [18]:
import os
import json
import numpy as np
import pandas as pd
import shap
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from statsmodels.stats.proportion import proportions_ztest

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import cm as rl_cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
    PageBreak, Table, TableStyle
)


def insightforge_dsmodule(
    file_path,
    sep=";",
    decimal=",",
    id_col="ID",
    target_score_col="score",
    threshold=10,
    min_profile_size=0.05,
    eps=0.25,
    output_path="IF_analytical_output"
):
    """
    Full analytical pipeline: SHAP importance, DBSCAN clustering,
    decision-tree profiling with statistical validation, odds-ratio,
    and PDF visual report.

    Parameters
    ----------
    file_path         : path to the input CSV file
    sep               : column separator in the CSV
    decimal           : decimal separator in the CSV
    id_col            : name of the unique identifier column
    target_score_col  : name of the model score column
    threshold         : percentile used to define HP (top) and LP (bottom)
    min_profile_size  : minimum profile size as a fraction of total rows (0-1)
                        applied both to DBSCAN min_samples and to leaf validation
    eps               : DBSCAN neighbourhood radius
    output_path       : folder where all output files are written

    Output files
    ------------
    shap_importance.json    global mean |SHAP| per feature, sorted descending
    PROFILE_RULES.csv       decision-tree path (rule) for each leaf / Profile_ID
    PROFILE_STATS.csv       mean of all numeric columns aggregated by Profile_ID
    ID_W_PROFILE.csv        row-level mapping: id, Profile_ID, HP, LP
    PROFILE_FEATURES.csv    decision-tree feature importances (non-zero only)
    oddsratio.csv           odds-ratio from multinomial logistic regression;
                            columns = original feature names, rows = Profile_ID classes
    InsightForge_Report.pdf visual report with SHAP chart, PCA scatter,
                            numeric and categorical profile charts
    """

    os.makedirs(output_path, exist_ok=True)

    # =========================================================
    # 1. LOAD DATA
    # =========================================================
    df = pd.read_csv(file_path, sep=sep, decimal=decimal)

    # =========================================================
    # 2. HP / LP / TRICT LABELS
    #    HP = top-threshold percentile, LP = bottom-threshold percentile
    #    TRICT: 0 = LP, 1 = mid, 2 = HP
    # =========================================================
    p_low, p_high = np.percentile(df[target_score_col], [threshold, 100 - threshold])

    df["HP"]    = np.where(df[target_score_col] >= p_high, 1, 0)
    df["LP"]    = np.where(df[target_score_col] <= p_low,  1, 0)
    df["TRICT"] = np.where(
        df[target_score_col] <= p_low,  0,
        np.where(df[target_score_col] > p_high, 2, 1)
    )

    # Global HP rate used as baseline for all propensity comparisons
    base_propensity = df["HP"].mean()

    # =========================================================
    # 3. FEATURE MATRIX
    #    Built once; reused for SHAP model, PCA/DBSCAN, and later
    #    as the base for the decision-tree and logistic regression.
    # =========================================================
    _internal_cols = {id_col, target_score_col, "HP", "LP", "TRICT"}

    raw_num_cols = [
        c for c in df.select_dtypes(exclude=["object", "category"]).columns
        if c not in _internal_cols
    ]
    raw_cat_cols = [
        c for c in df.select_dtypes(include=["object", "category"]).columns
        if c not in _internal_cols
    ]

    # Impute numerics with column mean
    X_num = df[raw_num_cols].fillna(df[raw_num_cols].mean())

    # One-hot encode categoricals (drop first to avoid multicollinearity)
    ohe_main = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
    if raw_cat_cols:
        X_cat_arr     = ohe_main.fit_transform(df[raw_cat_cols])
        ohe_cat_names = ohe_main.get_feature_names_out(raw_cat_cols).tolist()
        X_cat_df      = pd.DataFrame(X_cat_arr, columns=ohe_cat_names, index=df.index)
    else:
        X_cat_df      = pd.DataFrame(index=df.index)
        ohe_cat_names = []

    X                   = pd.concat([X_num, X_cat_df], axis=1)
    feature_names_model = list(X.columns)

    # Scale to [0, 1] for DBSCAN and logistic regression
    scaler   = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # =========================================================
    # 4. SHAP MODEL
    #    Select the best classifier (LightGBM vs RandomForest)
    #    by 5-fold CV ROC-AUC, then compute global SHAP importance.
    # =========================================================
    Y_hp = df["HP"].values

    search_gb = RandomizedSearchCV(
        lgb.LGBMClassifier(verbose=-1),
        {"n_estimators": [50, 100]},
        cv=5, scoring="roc_auc", n_jobs=-1, random_state=1926
    )
    search_rf = RandomizedSearchCV(
        RandomForestClassifier(),
        {"n_estimators": [50, 100]},
        cv=5, scoring="roc_auc", n_jobs=-1, random_state=1926
    )
    search_gb.fit(X_scaled, Y_hp)
    search_rf.fit(X_scaled, Y_hp)

    best_model = (
        search_gb if search_gb.best_score_ >= search_rf.best_score_
        else search_rf
    ).best_estimator_
    best_model.fit(X_scaled, Y_hp)

    explainer   = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_scaled)

    # Robust extraction: handle list output (RF) and 3-D arrays
    if isinstance(shap_values, list):
        shap_array = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    else:
        shap_array = shap_values
    if shap_array.ndim == 3:
        shap_array = shap_array[:, :, 1]

    mean_impact = np.abs(shap_array).mean(axis=0).reshape(-1)

    shap_output = sorted(
        [{"feature": feat, "mean_impact": float(mean_impact[i])}
         for i, feat in enumerate(feature_names_model)],
        key=lambda x: x["mean_impact"], reverse=True
    )
    with open(f"{output_path}/shap_importance.json", "w") as f:
        json.dump(shap_output, f, indent=2)

    # =========================================================
    # 5. PCA  (dimensionality reduction before DBSCAN)
    #    Retain enough components to explain 71% of variance.
    # =========================================================
    pca_cluster = PCA(n_components=0.71, random_state=1926)
    X_pca       = pca_cluster.fit_transform(X_scaled)

    # =========================================================
    # 6. DBSCAN  ->  initial cluster assignment
    #    min_samples enforces the minimum cluster size constraint.
    #    Noise points (label = -1) go to "NotHP" downstream.
    # =========================================================
    min_samples       = max(3, int(min_profile_size * len(X_pca)))
    db                = DBSCAN(eps=eps, min_samples=min_samples, metric="cosine")
    df["cluster"]     = db.fit_predict(X_pca)

    df["cluster_propensity"] = df.groupby("cluster")["HP"].transform("mean")

    # Pre-filter: retain clusters with propensity > 1.2x baseline
    df["final_cluster"] = np.where(
        df["cluster_propensity"] / base_propensity > 1.2,
        df["cluster"].astype(str),
        "Generic"
    )

    # =========================================================
    # 7. Z-TEST ON CLUSTERS
    #    Validate each candidate cluster with a two-proportion
    #    z-test (cluster HP rate vs rest of dataset HP rate).
    #    Clusters that fail are demoted to "NotHP".
    # =========================================================
    cluster_ztest_rows = []
    for cl in df["final_cluster"].unique():
        mask_cl = df["final_cluster"] == cl
        z_stat, p_val = proportions_ztest(
            count=[df.loc[mask_cl,  "HP"].sum(), df.loc[~mask_cl, "HP"].sum()],
            nobs =[int(mask_cl.sum()), int((~mask_cl).sum())]
        )
        cluster_ztest_rows.append({
            "final_cluster":   cl,
            "cluster_p_value": p_val,
            "cluster_z_stat":  z_stat
        })

    df = df.merge(pd.DataFrame(cluster_ztest_rows), on="final_cluster", how="left")

    df["final_cluster"] = np.where(
        (df["cluster_p_value"] <= 0.05) & (np.abs(df["cluster_z_stat"]) >= 1.96),
        df["final_cluster"],
        "Generic"
    )

    # =========================================================
    # 8. DECISION TREE  ->  leaf-level segmentation
    #    Target: final_cluster labels (validated DBSCAN clusters).
    #    The tree learns to reproduce cluster boundaries using the
    #    original features, producing interpretable rules.
    # =========================================================
    _drop_dt = {
        id_col, target_score_col,
        "HP", "LP", "TRICT",
        "cluster", "cluster_propensity",
        "final_cluster", "cluster_p_value", "cluster_z_stat"
    }
    X_dt_df = df.drop(columns=list(_drop_dt), errors="ignore")

    dt_num_cols = X_dt_df.select_dtypes(exclude=["object", "category"]).columns.tolist()
    dt_cat_cols = X_dt_df.select_dtypes(include=["object", "category"]).columns.tolist()

    ohe_dt = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
    if dt_cat_cols:
        X_dt_cat_arr = ohe_dt.fit_transform(X_dt_df[dt_cat_cols])
        dt_cat_names = ohe_dt.get_feature_names_out(dt_cat_cols).tolist()
        X_dt_cat_df  = pd.DataFrame(X_dt_cat_arr, columns=dt_cat_names, index=df.index)
    else:
        X_dt_cat_df  = pd.DataFrame(index=df.index)
        dt_cat_names = []

    X_dt          = pd.concat([X_dt_df[dt_num_cols], X_dt_cat_df], axis=1)
    dt_feat_names = list(X_dt.columns)
    X_dt_arr      = X_dt.fillna(X_dt.mean()).values
    Y_dt          = df["final_cluster"].values

    dt = DecisionTreeClassifier(random_state=1926)
    dt.fit(X_dt_arr, Y_dt)

    # Compute leaf-level HP propensity for each row
    df["leaf_id"]         = dt.apply(X_dt_arr)
    df["leaf_propensity"] = df.groupby("leaf_id")["HP"].transform("mean")

    # First pass: keep leaf as candidate profile if propensity > 1.2x baseline
    df["Profile_ID"] = np.where(
        df["leaf_propensity"] / base_propensity > 1.2,
        df["leaf_id"].astype(str),
        "Generic"
    )

    # =========================================================
    # 9. Z-TEST ON PROFILES  ->  Profile_ID final validation
    #    Each candidate Profile_ID is tested against the rest of
    #    the dataset.  Three conditions must ALL be satisfied:
    #      (a) leaf size >= min_profile_size * total rows
    #      (b) p-value <= 0.05
    #      (c) |z-stat| >= 1.96
    #    Any profile failing at least one condition becomes "Generic".
    # =========================================================
    min_leaf_size = int(min_profile_size * len(df))

    profile_ztest_rows = []
    for pid in df["Profile_ID"].unique():
        mask_pid   = df["Profile_ID"] == pid
        leaf_count = int(mask_pid.sum())
        z_stat, p_val = proportions_ztest(
            count=[df.loc[mask_pid,  "HP"].sum(), df.loc[~mask_pid, "HP"].sum()],
            nobs =[leaf_count, int((~mask_pid).sum())]
        )
        profile_ztest_rows.append({
            "Profile_ID":      pid,
            "profile_count":   leaf_count,
            "profile_p_value": p_val,
            "profile_z_stat":  z_stat
        })

    df = df.merge(pd.DataFrame(profile_ztest_rows), on="Profile_ID", how="left")

    df["Profile_ID"] = np.where(
        (df["profile_count"]   >= min_leaf_size) &
        (df["profile_p_value"] <= 0.05) &
        (np.abs(df["profile_z_stat"]) >= 1.96),
        df["Profile_ID"],
        "Generic"
    )

    # =========================================================
    # 10. RULE EXTRACTION
    #     Traverse the decision tree to recover the full
    #     conjunction of conditions that lead to each leaf node.
    # =========================================================
    def extract_rules(tree_clf, feat_names):
        cl = tree_clf.tree_.children_left
        cr = tree_clf.tree_.children_right
        ft = tree_clf.tree_.feature
        th = tree_clf.tree_.threshold
        rules = {}

        def recurse(node, path):
            if cl[node] == cr[node]:   # leaf node
                rules[node] = " AND ".join(path) if path else "ROOT"
            else:
                fname = feat_names[ft[node]]
                recurse(cl[node], path + [f"{fname} <= {th[node]:.3f}"])
                recurse(cr[node], path + [f"{fname} > {th[node]:.3f}"])

        recurse(0, [])
        return rules

    rules_dict = extract_rules(dt, dt_feat_names)

    df_rules = (
        df[["leaf_id", "Profile_ID"]]
        .drop_duplicates()
        .assign(rule=lambda d: d["leaf_id"].map(rules_dict))
        [["leaf_id", "Profile_ID", "rule"]]
        .sort_values("Profile_ID")
        .reset_index(drop=True)
    )
    df_rules.to_csv(f"{output_path}/PROFILE_RULES.csv", index=False)

    # =========================================================
    # 11. FEATURE IMPORTANCES  (decision tree, non-zero only)
    # =========================================================
    df_importances = (
        pd.DataFrame({"feature": dt_feat_names,
                      "importance": dt.feature_importances_})
        .query("importance > 0")
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    df_importances.to_csv(f"{output_path}/PROFILE_FEATURES.csv", index=False)

    # =========================================================
    # 12. PROFILE STATS  (mean of all numerics per Profile_ID)
    # =========================================================
    df["profile_volume"] = df.groupby("Profile_ID")[id_col].transform("count")
    df["profile_pct"]    = df["profile_volume"] / len(df)

    _exclude_agg = {
        "Profile_ID", id_col,
        "cluster", "leaf_id", "final_cluster",
        "cluster_p_value", "cluster_z_stat",
        "profile_p_value", "profile_z_stat", "profile_count"
    }
    agg_num_cols = [
        c for c in df.select_dtypes(include="number").columns
        if c not in _exclude_agg
    ]

    df_stats = (
        df.groupby("Profile_ID")[agg_num_cols]
        .mean()
        .reset_index()
    )
    df_stats.to_csv(f"{output_path}/PROFILE_STATS.csv", index=False)

    # =========================================================
    # 13. ID -> PROFILE TABLE
    # =========================================================
    df[[id_col, "Profile_ID", "HP", "LP"]].to_csv(
        f"{output_path}/ID_W_PROFILE.csv", index=False
    )

    # =========================================================
    # 14. ODDS RATIO  (multinomial logistic regression)
    #     Columns are the original feature names (readable).
    #     Each row corresponds to one Profile_ID class.
    # =========================================================
    _drop_lr = _drop_dt | {
        "Profile_ID",
        "cluster_propensity", "leaf_propensity",
        "profile_p_value", "profile_z_stat", "profile_count",
        "cluster_p_value", "cluster_z_stat",
        "profile_volume", "profile_pct"
    }
    X_lr_df = df.drop(columns=list(_drop_lr), errors="ignore")

    lr_num_cols = X_lr_df.select_dtypes(include=["number"]).columns.tolist()
    lr_cat_cols = X_lr_df.select_dtypes(exclude=["number"]).columns.tolist()

    X_lr_num = (
        SimpleImputer(strategy="mean").fit_transform(X_lr_df[lr_num_cols])
        if lr_num_cols else np.empty((len(X_lr_df), 0))
    )
    if lr_cat_cols:
        ohe_lr       = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
        X_lr_cat     = ohe_lr.fit_transform(X_lr_df[lr_cat_cols].fillna("missing"))
        lr_cat_names = ohe_lr.get_feature_names_out(lr_cat_cols).tolist()
    else:
        X_lr_cat     = np.empty((len(X_lr_df), 0))
        lr_cat_names = []

    lr_all_names = lr_num_cols + lr_cat_names
    X_lr_final   = MinMaxScaler().fit_transform(np.hstack([X_lr_num, X_lr_cat]))
    Y_lr         = df["Profile_ID"].values

    log = LogisticRegression(max_iter=5000, random_state=1926)
    log.fit(X_lr_final, Y_lr)

    # When only 2 classes exist sklearn returns coef_ with shape (1, n_features)
    # instead of (2, n_features). Expand symmetrically so both classes appear as rows.
    if len(log.classes_) == 2 and log.coef_.shape[0] == 1:
        coef_matrix = np.vstack([-log.coef_, log.coef_])
    else:
        coef_matrix = log.coef_

    df_or = pd.DataFrame(
        np.exp(coef_matrix),
        columns=lr_all_names,
        index=log.classes_
    )
    df_or.index.name = "Profile_ID"
    df_or.reset_index(inplace=True)
    df_or.to_csv(f"{output_path}/oddsratio.csv", index=False)

    # =========================================================
    # 15. CHARTS  (saved as temporary PNGs, embedded in PDF)
    # =========================================================
    plot_paths    = []
    profile_order = (
        sorted([p for p in df["Profile_ID"].unique() if p != "Generic"]) + ["Generic"]
    )

    # --- 15a. SHAP global feature importance bar chart (top 20) ---
    shap_df = pd.DataFrame(shap_output).head(20)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(shap_df["feature"][::-1], shap_df["mean_impact"][::-1], color="#4C72B0")
    ax.set_xlabel("Mean |SHAP| value")
    ax.set_title("Global SHAP Feature Importance (top 20)")
    plt.tight_layout()
    p_shap = f"{output_path}/_shap_bar.png"
    fig.savefig(p_shap, dpi=150)
    plt.close(fig)
    plot_paths.append(("Global SHAP Feature Importance", p_shap))

    # --- 15b. PCA 2D scatter coloured by Profile_ID ---
    pca2d   = PCA(n_components=2, random_state=1926)
    X_pca2d = pca2d.fit_transform(X_scaled)
    var_exp = pca2d.explained_variance_ratio_ * 100

    profiles_arr    = df["Profile_ID"].values
    unique_profiles = sorted(set(profiles_arr), key=lambda x: (x == "Generic", x))
    cmap_scatter    = cm.get_cmap("tab10", len(unique_profiles))
    profile_color   = {p: cmap_scatter(i) for i, p in enumerate(unique_profiles)}

    fig, ax = plt.subplots(figsize=(9, 6))
    for pid in unique_profiles:
        mask = profiles_arr == pid
        ax.scatter(
            X_pca2d[mask, 0], X_pca2d[mask, 1],
            c=[profile_color[pid]], label=str(pid),
            alpha=0.55, s=18, edgecolors="none"
        )
    ax.set_xlabel(f"PC1 ({var_exp[0]:.1f}% var)")
    ax.set_ylabel(f"PC2 ({var_exp[1]:.1f}% var)")
    ax.set_title("2D PCA Projection - coloured by Profile_ID")
    ax.legend(title="Profile_ID", bbox_to_anchor=(1.02, 1),
              loc="upper left", fontsize=7, markerscale=1.5)
    plt.tight_layout()
    p_pca2d = f"{output_path}/_pca2d_profiles.png"
    fig.savefig(p_pca2d, dpi=150)
    plt.close(fig)
    plot_paths.append(("2D PCA Projection by Profile_ID", p_pca2d))

    # --- 15c. Mean of each numeric variable per profile (bar chart) ---
    plot_num_cols = [
        c for c in raw_num_cols
        if c in df_stats.columns and c not in {"HP", "LP", "TRICT"}
    ]
    cmap_bar = cm.get_cmap("tab10", len(profile_order))

    for col in plot_num_cols[:10]:   # cap at 10 to keep the PDF manageable
        vals = [
            df_stats.loc[df_stats["Profile_ID"] == pid, col].values[0]
            if pid in df_stats["Profile_ID"].values else np.nan
            for pid in profile_order
        ]
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.bar(profile_order, vals,
               color=[cmap_bar(i) for i in range(len(profile_order))])
        ax.set_title(f"Avg '{col}' by Profile")
        ax.set_xlabel("Profile_ID")
        ax.set_ylabel(f"Mean {col}")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        p_num = f"{output_path}/_num_{col}.png"
        fig.savefig(p_num, dpi=150)
        plt.close(fig)
        plot_paths.append((f"Numeric: {col} by Profile", p_num))

    # --- 15d. Categorical distribution per profile (stacked bar, row %) ---
    for col in raw_cat_cols[:5]:     # cap at 5 categoricals
        ct = pd.crosstab(df["Profile_ID"], df[col], normalize="index")
        fig, ax = plt.subplots(figsize=(9, 4))
        ct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
        ax.set_title(f"'{col}' distribution by Profile (row %)")
        ax.set_xlabel("Profile_ID")
        ax.set_ylabel("Proportion")
        ax.legend(title=col, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        p_cat = f"{output_path}/_cat_{col}.png"
        fig.savefig(p_cat, dpi=150)
        plt.close(fig)
        plot_paths.append((f"Categorical: {col} by Profile", p_cat))

    # =========================================================
    # 16. PDF REPORT
    # =========================================================
    pdf_path = f"{output_path}/InsightForge_Report.pdf"
    doc      = SimpleDocTemplate(
        pdf_path, pagesize=A4,
        leftMargin=2*rl_cm, rightMargin=2*rl_cm,
        topMargin=2*rl_cm,  bottomMargin=2*rl_cm
    )
    styles   = getSampleStyleSheet()
    h1, h2, body = styles["Heading1"], styles["Heading2"], styles["Normal"]
    story    = []

    # Cover page
    story.append(Spacer(1, 2*rl_cm))
    story.append(Paragraph("InsightForge - Analytical Report", h1))
    story.append(Spacer(1, 0.5*rl_cm))
    story.append(Paragraph(
        f"File: <b>{os.path.basename(file_path)}</b> &nbsp;|&nbsp; "
        f"Score column: <b>{target_score_col}</b> &nbsp;|&nbsp; "
        f"Threshold: <b>{threshold}th pct</b> &nbsp;|&nbsp; "
        f"DBSCAN eps: <b>{eps}</b>",
        body
    ))
    story.append(Spacer(1, 0.4*rl_cm))

    # Profile summary table
    summary_data = [["Profile_ID", "N", "% Total", "HP Rate"]]
    for pid in profile_order:
        sub = df[df["Profile_ID"] == pid]
        summary_data.append([
            str(pid),
            str(len(sub)),
            f"{len(sub) / len(df) * 100:.1f}%",
            f"{sub['HP'].mean() * 100:.1f}%"
        ])

    tbl = Table(summary_data, hAlign="LEFT")
    tbl.setStyle(TableStyle([
        ("BACKGROUND",     (0, 0), (-1, 0),  colors.HexColor("#4C72B0")),
        ("TEXTCOLOR",      (0, 0), (-1, 0),  colors.white),
        ("FONTNAME",       (0, 0), (-1, 0),  "Helvetica-Bold"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.HexColor("#EEF2F8"), colors.white]),
        ("GRID",           (0, 0), (-1, -1), 0.5, colors.grey),
        ("FONTSIZE",       (0, 0), (-1, -1), 9),
        ("PADDING",        (0, 0), (-1, -1), 5),
    ]))
    story.append(tbl)
    story.append(PageBreak())

    # One chart per page section
    page_w = A4[0] - 4*rl_cm
    for title_txt, img_path in plot_paths:
        story.append(Paragraph(title_txt, h2))
        story.append(Spacer(1, 0.2*rl_cm))
        story.append(RLImage(img_path, width=page_w, height=page_w * 0.50))
        story.append(Spacer(1, 0.6*rl_cm))

    doc.build(story)

    # Remove temporary PNG files
    for _, img_path in plot_paths:
        try:
            os.remove(img_path)
        except OSError:
            pass

    print("PIPELINE COMPLETED SUCCESSFULLY")
    print(f"  -> Outputs saved in: {output_path}/")

    return {
        "shap_importance":  shap_output,
        "profile_stats":    df_stats,
        "profile_rules":    df_rules,
        "profile_features": df_importances,
        "oddsratio":        df_or,
        "id_with_profile":  df[[id_col, "Profile_ID", "HP", "LP"]],
        "pdf_report":       pdf_path,
    }


In [23]:
results = insightforge_dsmodule(
    file_path="IF_abt_input/ActionForge.csv",        # path al tuo CSV
    sep=";",                     # separatore colonne
    decimal=",",                 # separatore decimale
    id_col="IDCLI",                 # colonna identificativo univoco
    target_score_col="score",    # colonna con lo score del modello
    threshold=15,                # percentile per definire HP/LP (es. top/bottom 10%)
    min_profile_size=0.05,       # dimensione minima cluster (5% del totale)
    eps=0.2,                    # raggio neighbourhood DBSCAN
    output_path="IF_analytical_output"      # cartella dove salvare tutti gli output
)

# Accedere agli output in memoria:
results["shap_importance"]   # lista dizionari con feature e mean_impact
results["profile_stats"]     # DataFrame statistiche aggregate per profilo
results["profile_rules"]     # DataFrame regole decision tree
results["profile_features"]  # DataFrame importanze feature
results["oddsratio"]         # DataFrame odds ratio logistica multinomiale
results["id_with_profile"]   # DataFrame id → Profile_ID, HP, LP
results["pdf_report"]        # path al PDF generato

C:\Users\luigi\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\luigi\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 2 is smaller than n_iter=10. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
C:\Users\luigi\anaconda3\Lib\site-packages\shap\explainers\_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
C:\Users\luigi\AppData\Local\Temp\ipykernel_23012\3689122924.py:471: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_scatter    = cm.get_cmap("t

PIPELINE COMPLETED SUCCESSFULLY
  -> Outputs saved in: IF_analytical_output/


'IF_analytical_output/InsightForge_Report.pdf'

'C:\\Users\\luigi\\InsightForge'